In [1]:
import os
import json
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path
from typing import Dict, List, Any
import ipywidgets as widgets
from IPython.display import display, HTML

# 設定結果資料夾路徑
RESULTS_DIR = Path('.')
print(f"Results directory: {RESULTS_DIR.absolute()}")

Results directory: /home/claire/Documents/Green_AI/results


## 1. 資料載入與探索

In [2]:
def scan_available_experiments():
    """掃描所有可用的實驗"""
    experiments = {
        'baseline_models': [],
        'quantized_models': [],
        'optimization_runs': []
    }
    
    # 掃描基準模型
    for item in RESULTS_DIR.iterdir():
        if item.is_dir():
            name = item.name
            # 基準模型（沒有量化後綴）
            if 'Llama' in name and not any(x in name for x in ['awq', 'bnb', 'gptq']):
                if (item / 'gsm8k_results.json').exists():
                    experiments['baseline_models'].append(name)
            # 量化模型
            elif 'Llama' in name and any(x in name for x in ['awq', 'bnb', 'gptq']):
                if (item / 'gsm8k_results.json').exists():
                    experiments['quantized_models'].append(name)
    
    # 掃描優化實驗
    opt_dir = RESULTS_DIR / 'optimization'
    if opt_dir.exists():
        for item in opt_dir.iterdir():
            if item.is_dir() and (item / 'summary.json').exists():
                experiments['optimization_runs'].append(item.name)
    
    return experiments

# 掃描實驗
experiments = scan_available_experiments()
print("\n=== 可用實驗 ===")
print(f"\n基準模型 ({len(experiments['baseline_models'])}):", experiments['baseline_models'])
print(f"\n量化模型 ({len(experiments['quantized_models'])}):", experiments['quantized_models'][:5], '...')
print(f"\n優化實驗 ({len(experiments['optimization_runs'])}):", experiments['optimization_runs'])


=== 可用實驗 ===

基準模型 (3): ['Llama-3.2-3B-Instruct', 'Llama-3.1-8B-Instruct', 'Llama-3.2-1B-Instruct']

量化模型 (9): ['Llama-3.2-3B-Instruct-bnb-4bit-20251118_030711', 'Llama-3.2-3B-Instruct-awq-4bit-20251109_012853', 'Llama-3.2-1B-Instruct-bnb-4bit-20251116_225823', 'Llama-3.1-8B-Instruct-awq-4bit-20251115_162005', 'Llama-3.2-1B-Instruct-gptq-4bit-20251019_163501'] ...

優化實驗 (5): ['quick-mo-test_20251202_221816', 'quick-mo-test_20251127_165523', 'quick-mo-test_20251130_145632', 'quick-mo-test_20251130_161230', 'quick-mo-test_20251203_161824']


In [3]:
def load_model_results(model_name: str, dataset: str = 'gsm8k') -> Dict:
    """載入模型結果"""
    result_file = RESULTS_DIR / model_name / f"{dataset}_results.json"
    if result_file.exists():
        with open(result_file, 'r') as f:
            return json.load(f)
    return None

def load_optimization_results(run_name: str) -> Dict:
    """載入優化實驗結果"""
    run_dir = RESULTS_DIR / 'optimization' / run_name
    results = {}
    
    files = ['summary.json', 'pareto_frontier.json', 'all_trials.json']
    for file in files:
        file_path = run_dir / file
        if file_path.exists():
            with open(file_path, 'r') as f:
                results[file.replace('.json', '')] = json.load(f)
    
    return results

## 2. 基準模型與量化模型比較

In [4]:
# 選擇要比較的模型
print("選擇基準模型和量化模型進行比較：\n")

baseline_dropdown = widgets.Dropdown(
    options=experiments['baseline_models'],
    description='基準模型:',
    style={'description_width': '100px'}
)

quantized_multiselect = widgets.SelectMultiple(
    options=experiments['quantized_models'],
    description='量化模型:',
    rows=8,
    style={'description_width': '100px'}
)

dataset_dropdown = widgets.Dropdown(
    options=['gsm8k', 'truthfulqa', 'commonsenseqa', 'bbh', 'humaneval'],
    description='資料集:',
    value='gsm8k',
    style={'description_width': '100px'}
)

compare_button = widgets.Button(
    description='生成比較圖表',
    button_style='primary',
    icon='chart-bar'
)

output_area = widgets.Output()

display(baseline_dropdown, quantized_multiselect, dataset_dropdown, compare_button, output_area)

選擇基準模型和量化模型進行比較：



Dropdown(description='基準模型:', options=('Llama-3.2-3B-Instruct', 'Llama-3.1-8B-Instruct', 'Llama-3.2-1B-Instruc…

SelectMultiple(description='量化模型:', options=('Llama-3.2-3B-Instruct-bnb-4bit-20251118_030711', 'Llama-3.2-3B-I…

Dropdown(description='資料集:', options=('gsm8k', 'truthfulqa', 'commonsenseqa', 'bbh', 'humaneval'), style=Descr…

Button(button_style='primary', description='生成比較圖表', icon='chart-bar', style=ButtonStyle())

Output()

In [5]:
def plot_model_comparison(baseline_name: str, quantized_names: List[str], dataset: str):
    """繪製模型比較圖"""
    
    # 載入資料
    baseline_data = load_model_results(baseline_name, dataset)
    if not baseline_data:
        print(f"無法載入基準模型資料: {baseline_name}")
        return
    
    quantized_data = []
    model_names = [baseline_name]
    for name in quantized_names:
        data = load_model_results(name, dataset)
        if data:
            quantized_data.append(data)
            model_names.append(name)
    
    if not quantized_data:
        print("無法載入量化模型資料")
        return
    
    all_data = [baseline_data] + quantized_data
    
    # 提取指標
    accuracies = [d['accuracy'] * 100 for d in all_data]
    gpu_peaks = [d['gpu_peak_mb'] for d in all_data]
    throughputs = [d.get('throughput_tokens_per_sec', 0) for d in all_data]
    
    # 計算相對變化（相對於基準）
    acc_changes = [(a - accuracies[0]) for a in accuracies]
    gpu_changes = [(g - gpu_peaks[0]) / gpu_peaks[0] * 100 for g in gpu_peaks]
    throughput_changes = [(t - throughputs[0]) / throughputs[0] * 100 if throughputs[0] > 0 else 0 for t in throughputs]
    
    # 建立子圖
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            f'{dataset.upper()} 準確率',
            'GPU 記憶體峰值',
            '準確率變化 vs GPU 變化',
            '吞吐量 (tokens/sec)'
        ),
        specs=[
            [{"type": "bar"}, {"type": "bar"}],
            [{"type": "scatter"}, {"type": "bar"}]
        ]
    )
    
    # 顏色：基準=藍色，量化=綠色系
    colors = ['#1f77b4'] + ['#2ca02c'] * len(quantized_names)
    
    # 1. 準確率
    fig.add_trace(
        go.Bar(
            x=[n.split('/')[-1][:20] for n in model_names],
            y=accuracies,
            marker_color=colors,
            text=[f"{a:.2f}%" for a in accuracies],
            textposition='outside',
            name='準確率'
        ),
        row=1, col=1
    )
    
    # 2. GPU 記憶體
    fig.add_trace(
        go.Bar(
            x=[n.split('/')[-1][:20] for n in model_names],
            y=gpu_peaks,
            marker_color=colors,
            text=[f"{g:.0f} MB" for g in gpu_peaks],
            textposition='outside',
            name='GPU 峰值'
        ),
        row=1, col=2
    )
    
    # 3. 準確率 vs GPU 變化散點圖
    fig.add_trace(
        go.Scatter(
            x=gpu_changes,
            y=acc_changes,
            mode='markers+text',
            marker=dict(size=12, color=colors),
            text=[n.split('/')[-1][:15] for n in model_names],
            textposition='top center',
            name='模型',
            hovertemplate='<b>%{text}</b><br>' +
                         'GPU 變化: %{x:.2f}%<br>' +
                         '準確率變化: %{y:.2f}%<extra></extra>'
        ),
        row=2, col=1
    )
    
    # 添加參考線（原點）
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5, row=2, col=1)
    fig.add_vline(x=0, line_dash="dash", line_color="gray", opacity=0.5, row=2, col=1)
    
    # 4. 吞吐量
    fig.add_trace(
        go.Bar(
            x=[n.split('/')[-1][:20] for n in model_names],
            y=throughputs,
            marker_color=colors,
            text=[f"{t:.0f}" for t in throughputs],
            textposition='outside',
            name='吞吐量'
        ),
        row=2, col=2
    )
    
    # 更新佈局
    fig.update_xaxes(tickangle=45)
    fig.update_yaxes(title_text="準確率 (%)", row=1, col=1)
    fig.update_yaxes(title_text="GPU 記憶體 (MB)", row=1, col=2)
    fig.update_xaxes(title_text="GPU 峰值變化 (%)", row=2, col=1)
    fig.update_yaxes(title_text="準確率變化 (%)", row=2, col=1)
    fig.update_yaxes(title_text="Tokens/sec", row=2, col=2)
    
    fig.update_layout(
        title_text=f"模型比較：{baseline_name} vs 量化版本 ({dataset.upper()})",
        height=800,
        showlegend=False
    )
    
    fig.show()
    
    # 顯示統計表格
    df = pd.DataFrame({
        '模型': [n.split('/')[-1] for n in model_names],
        '準確率': [f"{a:.2f}%" for a in accuracies],
        '準確率變化': [f"{a:+.2f}%" for a in acc_changes],
        'GPU 峰值 (MB)': [f"{g:.0f}" for g in gpu_peaks],
        'GPU 變化': [f"{g:+.2f}%" for g in gpu_changes],
        '吞吐量': [f"{t:.0f}" for t in throughputs],
        '吞吐量變化': [f"{t:+.2f}%" for t in throughput_changes]
    })
    
    print("\n=== 詳細指標 ===")
    display(df)

def on_compare_clicked(b):
    output_area.clear_output()
    with output_area:
        baseline = baseline_dropdown.value
        quantized = list(quantized_multiselect.value)
        dataset = dataset_dropdown.value
        
        if not baseline:
            print("請選擇基準模型")
            return
        if not quantized:
            print("請選擇至少一個量化模型")
            return
        
        plot_model_comparison(baseline, quantized, dataset)

compare_button.on_click(on_compare_clicked)

## 3. 優化實驗結果視覺化

In [6]:
# 選擇優化實驗
print("選擇優化實驗進行視覺化：\n")

optimization_dropdown = widgets.Dropdown(
    options=experiments['optimization_runs'],
    description='實驗:',
    style={'description_width': '100px'}
)

viz_button = widgets.Button(
    description='生成視覺化',
    button_style='success',
    icon='chart-line'
)

opt_output_area = widgets.Output()

display(optimization_dropdown, viz_button, opt_output_area)

選擇優化實驗進行視覺化：



Dropdown(description='實驗:', options=('quick-mo-test_20251202_221816', 'quick-mo-test_20251127_165523', 'quick-…

Button(button_style='success', description='生成視覺化', icon='chart-line', style=ButtonStyle())

Output()

In [7]:
def plot_optimization_results(run_name: str):
    """繪製優化實驗結果"""
    
    results = load_optimization_results(run_name)
    if not results:
        print(f"無法載入優化實驗資料: {run_name}")
        return
    
    summary = results.get('summary', {})
    pareto = results.get('pareto_frontier', {})
    all_trials = results.get('all_trials', {})
    
    # 顯示摘要資訊
    print("\n" + "="*60)
    print(f"實驗名稱: {summary.get('experiment_name', 'N/A')}")
    print(f"時間戳記: {summary.get('timestamp', 'N/A')}")
    print(f"總試驗數: {summary.get('optimization', {}).get('total_trials', 'N/A')}")
    print(f"Pareto 解數量: {summary.get('optimization', {}).get('pareto_solutions', 'N/A')}")
    print(f"滿足目標數: {summary.get('optimization', {}).get('satisfying_solutions', 'N/A')}")
    print("="*60 + "\n")
    
    # 基準資訊
    baseline = summary.get('baseline', {})
    print(f"\n基準模型: {baseline.get('model', 'N/A')}")
    print(f"  準確率: {baseline.get('accuracy', 0)*100:.2f}%")
    print(f"  GPU 峰值: {baseline.get('gpu_peak_mb', 0):.0f} MB")
    print(f"  平均延遲: {baseline.get('avg_latency_ms', 0):.2f} ms")
    
    # 推薦配置
    recommended = summary.get('recommended', {})
    if recommended:
        print(f"\n推薦配置:")
        print(f"  方法: {recommended.get('method', 'N/A')}")
        obj = recommended.get('objectives', {})
        print(f"  準確率變化: {obj.get('accuracy_change', 0)*100:+.2f}%")
        print(f"  GPU 峰值變化: {obj.get('gpu_peak_change', 0)*100:+.2f}%")
        print(f"  延遲變化: {obj.get('latency_change', 0)*100:+.2f}%")
        print(f"  是否滿足目標: {'✓' if recommended.get('satisfies_targets') else '✗'}")
    
    # ===== 圖表 1: 3D Pareto 前沿 =====
    if 'pareto_solutions' in pareto:
        pareto_trials = pareto['pareto_solutions']
        
        acc_changes = [t['objectives']['accuracy_change'] * 100 for t in pareto_trials]
        gpu_changes = [t['objectives']['gpu_peak_change'] * 100 for t in pareto_trials]
        latency_changes = [t['objectives']['latency_change'] * 100 for t in pareto_trials]
        methods = [t['config']['method'] for t in pareto_trials]
        satisfies = [t.get('satisfies_targets', False) for t in pareto_trials]
        
        colors = ['green' if s else 'blue' for s in satisfies]
        
        fig = go.Figure(data=[go.Scatter3d(
            x=acc_changes,
            y=gpu_changes,
            z=latency_changes,
            mode='markers+text',
            marker=dict(size=8, color=colors, opacity=0.8),
            text=methods,
            textposition='top center',
            hovertemplate='<b>Method: %{text}</b><br>' +
                         '準確率變化: %{x:+.2f}%<br>' +
                         'GPU 峰值變化: %{y:+.2f}%<br>' +
                         '延遲變化: %{z:+.2f}%<extra></extra>'
        )])
        
        fig.update_layout(
            title='3D Pareto 前沿 (藍色=普通, 綠色=滿足目標)',
            scene=dict(
                xaxis_title='準確率變化 (%) [正=上升↑]',
                yaxis_title='GPU 峰值變化 (%) [負=減少↓]',
                zaxis_title='延遲變化 (%) [負=加速↓]'
            ),
            width=900,
            height=700
        )
        
        fig.show()
    
    # ===== 圖表 2: 方法比較矩陣 =====
    if 'trials' in all_trials:
        trials = all_trials['trials']
        
        # 按方法分組
        method_groups = {}
        for t in trials:
            if 'objectives' in t and all(k in t['objectives'] for k in ['accuracy_change', 'gpu_peak_change', 'latency_change']):
                method = t['config']['method']
                if method not in method_groups:
                    method_groups[method] = []
                method_groups[method].append(t)
        
        # 計算每個方法的平均表現
        method_stats = []
        for method, trials_list in method_groups.items():
            avg_acc = sum(t['objectives']['accuracy_change'] for t in trials_list) / len(trials_list) * 100
            avg_gpu = sum(t['objectives']['gpu_peak_change'] for t in trials_list) / len(trials_list) * 100
            avg_lat = sum(t['objectives']['latency_change'] for t in trials_list) / len(trials_list) * 100
            method_stats.append({
                'method': method,
                'avg_acc': avg_acc,
                'avg_gpu': avg_gpu,
                'avg_lat': avg_lat,
                'count': len(trials_list)
            })
        
        df_methods = pd.DataFrame(method_stats)
        
        # 建立比較圖
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=(
                '平均準確率變化',
                '平均 GPU 峰值變化',
                '平均延遲變化',
                '試驗次數分佈'
            )
        )
        
        # 準確率
        fig.add_trace(
            go.Bar(x=df_methods['method'], y=df_methods['avg_acc'], 
                  marker_color='lightblue', name='準確率'),
            row=1, col=1
        )
        
        # GPU
        fig.add_trace(
            go.Bar(x=df_methods['method'], y=df_methods['avg_gpu'],
                  marker_color='lightgreen', name='GPU'),
            row=1, col=2
        )
        
        # 延遲
        fig.add_trace(
            go.Bar(x=df_methods['method'], y=df_methods['avg_lat'],
                  marker_color='lightcoral', name='延遲'),
            row=2, col=1
        )
        
        # 試驗次數
        fig.add_trace(
            go.Bar(x=df_methods['method'], y=df_methods['count'],
                  marker_color='lightyellow', name='次數'),
            row=2, col=2
        )
        
        fig.update_yaxes(title_text="變化 (%)", row=1, col=1)
        fig.update_yaxes(title_text="變化 (%)", row=1, col=2)
        fig.update_yaxes(title_text="變化 (%)", row=2, col=1)
        fig.update_yaxes(title_text="試驗次數", row=2, col=2)
        
        fig.update_layout(
            title_text="量化方法平均表現比較",
            height=700,
            showlegend=False
        )
        
        fig.show()
        
        # 顯示統計表
        print("\n=== 方法統計 ===")
        df_display = df_methods.copy()
        df_display['avg_acc'] = df_display['avg_acc'].apply(lambda x: f"{x:+.2f}%")
        df_display['avg_gpu'] = df_display['avg_gpu'].apply(lambda x: f"{x:+.2f}%")
        df_display['avg_lat'] = df_display['avg_lat'].apply(lambda x: f"{x:+.2f}%")
        df_display.columns = ['方法', '平均準確率變化', '平均GPU變化', '平均延遲變化', '試驗次數']
        display(df_display)
    
    # ===== 圖表 3: 權衡散點圖矩陣 =====
    if 'pareto_solutions' in pareto:
        pareto_trials = pareto['pareto_solutions']
        
        acc_changes = [t['objectives']['accuracy_change'] * 100 for t in pareto_trials]
        gpu_changes = [t['objectives']['gpu_peak_change'] * 100 for t in pareto_trials]
        latency_changes = [t['objectives']['latency_change'] * 100 for t in pareto_trials]
        methods = [t['config']['method'] for t in pareto_trials]
        
        fig = make_subplots(
            rows=1, cols=3,
            subplot_titles=('準確率 vs GPU', '準確率 vs 延遲', 'GPU vs 延遲')
        )
        
        # Acc vs GPU
        fig.add_trace(
            go.Scatter(x=gpu_changes, y=acc_changes, mode='markers+text',
                      marker=dict(size=10), text=methods, textposition='top center',
                      name=''),
            row=1, col=1
        )
        fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5, row=1, col=1)
        fig.add_vline(x=0, line_dash="dash", line_color="gray", opacity=0.5, row=1, col=1)
        
        # Acc vs Latency
        fig.add_trace(
            go.Scatter(x=latency_changes, y=acc_changes, mode='markers+text',
                      marker=dict(size=10), text=methods, textposition='top center',
                      name=''),
            row=1, col=2
        )
        fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5, row=1, col=2)
        fig.add_vline(x=0, line_dash="dash", line_color="gray", opacity=0.5, row=1, col=2)
        
        # GPU vs Latency
        fig.add_trace(
            go.Scatter(x=latency_changes, y=gpu_changes, mode='markers+text',
                      marker=dict(size=10), text=methods, textposition='top center',
                      name=''),
            row=1, col=3
        )
        fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5, row=1, col=3)
        fig.add_vline(x=0, line_dash="dash", line_color="gray", opacity=0.5, row=1, col=3)
        
        fig.update_xaxes(title_text="GPU 峰值變化 (%)", row=1, col=1)
        fig.update_xaxes(title_text="延遲變化 (%)", row=1, col=2)
        fig.update_xaxes(title_text="延遲變化 (%)", row=1, col=3)
        fig.update_yaxes(title_text="準確率變化 (%)", row=1, col=1)
        fig.update_yaxes(title_text="準確率變化 (%)", row=1, col=2)
        fig.update_yaxes(title_text="GPU 峰值變化 (%)", row=1, col=3)
        
        fig.update_layout(
            title_text="Pareto 前沿權衡分析",
            height=400,
            width=1200,
            showlegend=False
        )
        
        fig.show()

def on_viz_clicked(b):
    opt_output_area.clear_output()
    with opt_output_area:
        run_name = optimization_dropdown.value
        if not run_name:
            print("請選擇優化實驗")
            return
        plot_optimization_results(run_name)

viz_button.on_click(on_viz_clicked)

## 4. 跨模型大小比較

比較不同模型大小（1B, 3B, 8B）在相同量化方法下的表現

In [ ]:
def plot_cross_model_comparison(dataset: str = 'gsm8k'):
    """跨模型大小比較"""
    
    # 定義要比較的模型
    model_configs = [
        ('Llama-3.2-1B-Instruct', '1B 基準'),
        ('Llama-3.2-1B-Instruct-awq-4bit-20251116_010732', '1B AWQ'),
        ('Llama-3.2-1B-Instruct-bnb-4bit-20251116_225823', '1B BNB'),
        ('Llama-3.2-1B-Instruct-gptq-4bit-20251019_163501', '1B GPTQ'),
        ('Llama-3.2-3B-Instruct', '3B 基準'),
        ('Llama-3.2-3B-Instruct-awq-4bit-20251109_012853', '3B AWQ'),
        ('Llama-3.2-3B-Instruct-bnb-4bit-20251118_030711', '3B BNB'),
        ('Llama-3.2-3B-Instruct-gptq-4bit-20251109_203802', '3B GPTQ'),
        ('Llama-3.1-8B-Instruct', '8B 基準'),
        ('Llama-3.1-8B-Instruct-awq-4bit-20251115_162005', '8B AWQ'),
        ('Llama-3.1-8B-Instruct-bnb-4bit-20251118_115157', '8B BNB'),
        ('Llama-3.1-8B-Instruct-gptq-4bit-20251019_164317', '8B GPTQ'),
    ]
    
    # 載入資料
    data = []
    for model_name, label in model_configs:
        result = load_model_results(model_name, dataset)
        if result:
            data.append({
                'label': label,
                'model_name': model_name,
                'accuracy': result['accuracy'] * 100,
                'gpu_peak': result['gpu_peak_mb'],
                'throughput': result.get('throughput_tokens_per_sec', 0)
            })
    
    if not data:
        print("無法載入模型資料")
        return
    
    df = pd.DataFrame(data)
    
    # 建立圖表
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('準確率比較', 'GPU 記憶體比較', '吞吐量比較', '效率分數'),
        specs=[
            [{"type": "bar"}, {"type": "bar"}],
            [{"type": "bar"}, {"type": "scatter"}]
        ]
    )
    
    # 顏色編碼：基準=藍，AWQ=綠，BNB=橙，GPTQ=紅
    colors = []
    for label in df['label']:
        if '基準' in label:
            colors.append('#1f77b4')
        elif 'AWQ' in label:
            colors.append('#2ca02c')
        elif 'BNB' in label:
            colors.append('#ff7f0e')
        else:  # GPTQ
            colors.append('#d62728')
    
    # 1. 準確率
    fig.add_trace(
        go.Bar(x=df['label'], y=df['accuracy'], marker_color=colors,
              text=[f"{a:.2f}%" for a in df['accuracy']], textposition='outside'),
        row=1, col=1
    )
    
    # 2. GPU
    fig.add_trace(
        go.Bar(x=df['label'], y=df['gpu_peak'], marker_color=colors,
              text=[f"{g:.0f}" for g in df['gpu_peak']], textposition='outside'),
        row=1, col=2
    )
    
    # 3. 吞吐量
    fig.add_trace(
        go.Bar(x=df['label'], y=df['throughput'], marker_color=colors,
              text=[f"{t:.0f}" for t in df['throughput']], textposition='outside'),
        row=2, col=1
    )
    
    # 4. 效率分數 (準確率 / GPU 峰值)
    efficiency = df['accuracy'] / (df['gpu_peak'] / 1000)  # 每 GB 的準確率
    fig.add_trace(
        go.Scatter(x=df['gpu_peak'], y=df['accuracy'], mode='markers+text',
                  marker=dict(size=10, color=colors), text=df['label'],
                  textposition='top center'),
        row=2, col=2
    )
    
    fig.update_xaxes(tickangle=45)
    fig.update_yaxes(title_text="準確率 (%)", row=1, col=1)
    fig.update_yaxes(title_text="GPU 記憶體 (MB)", row=1, col=2)
    fig.update_yaxes(title_text="Tokens/sec", row=2, col=1)
    fig.update_xaxes(title_text="GPU 記憶體 (MB)", row=2, col=2)
    fig.update_yaxes(title_text="準確率 (%)", row=2, col=2)
    
    fig.update_layout(
        title_text=f"跨模型大小比較 ({dataset.upper()})",
        height=800,
        showlegend=False
    )
    
    fig.show()
    
    # 顯示效率表
    df_display = df.copy()
    df_display['效率分數'] = efficiency
    df_display['準確率'] = df_display['accuracy'].apply(lambda x: f"{x:.2f}%")
    df_display['GPU峰值'] = df_display['gpu_peak'].apply(lambda x: f"{x:.0f} MB")
    df_display['吞吐量'] = df_display['throughput'].apply(lambda x: f"{x:.0f}")
    df_display['效率分數'] = df_display['效率分數'].apply(lambda x: f"{x:.2f}")
    
    print("\n=== 跨模型效率比較 ===")
    display(df_display[['label', '準確率', 'GPU峰值', '吞吐量', '效率分數']])

# 建立按鈕
cross_dataset_dropdown = widgets.Dropdown(
    options=['gsm8k', 'truthfulqa', 'commonsenseqa', 'bbh', 'humaneval'],
    description='資料集:',
    value='gsm8k'
)

cross_button = widgets.Button(
    description='生成跨模型比較',
    button_style='info',
    icon='balance-scale'
)

cross_output_area = widgets.Output()

def on_cross_clicked(b):
    cross_output_area.clear_output()
    with cross_output_area:
        plot_cross_model_comparison(cross_dataset_dropdown.value)

cross_button.on_click(on_cross_clicked)

display(cross_dataset_dropdown, cross_button, cross_output_area)

Dropdown(description='資料集:', options=('gsm8k', 'truthfulqa', 'commonsenseqa', 'bbh', 'humaneval'), value='gsm8…

Button(button_style='info', description='生成跨模型比較', icon='balance-scale', style=ButtonStyle())

Output()

## 5. 自訂查詢與分析

使用下面的程式碼區塊進行自訂分析

In [9]:
# 範例：載入特定模型並檢視詳細資料
model_name = "Llama-3.2-1B-Instruct"
dataset = "gsm8k"

data = load_model_results(model_name, dataset)
if data:
    print(f"模型: {model_name}")
    print(f"資料集: {dataset}")
    print(f"準確率: {data['accuracy']*100:.2f}%")
    print(f"GPU 峰值: {data['gpu_peak_mb']:.0f} MB")
    print(f"總樣本數: {data['total']}")
    print(f"正確數: {data['correct']}")

模型: Llama-3.2-1B-Instruct
資料集: gsm8k
準確率: 33.89%
GPU 峰值: 2482 MB
總樣本數: 1319
正確數: 447


In [10]:
# 範例：比較特定量化方法在所有模型大小上的表現
def compare_quantization_method(method: str, dataset: str = 'gsm8k'):
    """比較特定量化方法在不同模型大小上的表現"""
    
    models = [
        f"Llama-3.2-1B-Instruct-{method}",
        f"Llama-3.2-3B-Instruct-{method}",
        f"Llama-3.1-8B-Instruct-{method}"
    ]
    
    # 嘗試找到匹配的模型
    found_models = []
    for pattern in models:
        for model in experiments['quantized_models']:
            if pattern in model:
                found_models.append(model)
                break
    
    if not found_models:
        print(f"找不到使用 {method} 方法的模型")
        return
    
    print(f"\n比較量化方法: {method.upper()}\n")
    
    results = []
    for model in found_models:
        data = load_model_results(model, dataset)
        if data:
            size = '1B' if '1B' in model else ('3B' if '3B' in model else '8B')
            results.append({
                'size': size,
                'accuracy': data['accuracy'] * 100,
                'gpu_peak': data['gpu_peak_mb'],
                'throughput': data.get('throughput_tokens_per_sec', 0)
            })
    
    df = pd.DataFrame(results)
    display(df)
    
    # 視覺化
    fig = go.Figure()
    fig.add_trace(go.Bar(x=df['size'], y=df['accuracy'], name='準確率 (%)'))
    fig.update_layout(title=f"{method.upper()} 量化方法 - 準確率比較", xaxis_title='模型大小', yaxis_title='準確率 (%)')
    fig.show()

# 使用範例
compare_quantization_method('awq-4bit', 'gsm8k')


比較量化方法: AWQ-4BIT



,size,accuracy,gpu_peak,throughput
0,1B,25.852919,1128.1133,1634.2932
1,3B,67.399545,2414.0664,731.8588
2,8B,73.919636,5831.3184,589.3546


## 6. 匯出報告

將視覺化結果匯出為 HTML 報告

In [ ]:
def export_html_report(optimization_run: str = None):
    """匯出 HTML 報告"""
    from datetime import datetime
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    report_file = RESULTS_DIR / f"visualization_report_{timestamp}.html"
    
    html_content = f"""
    <html>
    <head>
        <title>Green AI 實驗報告</title>
        <style>
            body {{ font-family: Arial, sans-serif; margin: 20px; }}
            h1 {{ color: #2c3e50; }}
            h2 {{ color: #34495e; }}
            .info {{ background-color: #ecf0f1; padding: 10px; margin: 10px 0; }}
        </style>
    </head>
    <body>
        <h1>Green AI 量化實驗報告</h1>
        <div class="info">
            <p>生成時間: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}</p>
            <p>基準模型數: {len(experiments['baseline_models'])}</p>
            <p>量化模型數: {len(experiments['quantized_models'])}</p>
            <p>優化實驗數: {len(experiments['optimization_runs'])}</p>
        </div>
        
        <h2>使用說明</h2>
        <p>請在 Jupyter Notebook 中使用互動式介面進行視覺化分析。</p>
        <p>此報告為基本摘要，詳細圖表請參考 Notebook。</p>
    </body>
    </html>
    """
    
    with open(report_file, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    print(f"\n報告已匯出至: {report_file}")

# 匯出報告
# export_html_report()